In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Script 1 — Keyword → Error-Variant Map Construction

Goal:
    Build a canonical keyword map from the manually annotated gold file
    (400_annotated_complete.csv). This map is the backbone for Scripts 2 and 3.

Inputs:
    - 400_annotated_complete.csv
        Required columns (names may vary slightly, we normalize):
            * Sub-Subtype    : fine-grained error type label (22 sub-subtypes)
            * Keyword        : canonical phrase that SHOULD appear in a correct translation
            * Best_Match     : attested error variant in MT output
            * Sentence_ID    : used only for sanity stats

Outputs:
    - keyword_map.json
        {
          "Definiteness Shift": {
            "<keyword_1>": "<error_variant_1 or '[OMITTED]'>",
            "<keyword_2>": "<error_variant_2>",
            ...
          },
          "Total Omission": {
            "<keyword_3>": "[OMITTED]",
            ...
          },
          ...
        }

    - keyword_map_for_gen.json
        {
          "Definiteness Shift": [
            {"keyword": "<keyword_1>", "best_match": "<error_variant_1>"},
            ...
          ],
          "Total Omission": [
            {"keyword": "<keyword_3>", "best_match": "[OMITTED]"},
            ...
          ]
        }

Semantics (frozen for all downstream scripts):
    - Keyword     = canonical phrase (correct lexical trigger).
    - Best_Match  = error variant (mutated phrase) for all non-omission types.
    - Total Omission:
        * CSV may have arbitrary/empty Best_Match.
        * In keyword_map, Best_Match is normalized to sentinel "[OMITTED]".
"""

import json
from collections import defaultdict

import pandas as pd

CSV_PATH = "400_annotated_complete.csv"
KEYWORD_MAP_JSON = "keyword_map.json"
KEYWORD_MAP_FOR_GEN_JSON = "keyword_map_for_gen.json"

# Mapping from raw PDF/CSV labels to canonical script labels
PDF_TO_SCRIPT_NAMES = {
    "Meaning Shift": "Meaning Shift",
    "Total Omission": "Total Omission",
    "Total Omission ": "Total Omission",
    "Partial Translation": "Partial Translation",

    "Literal Translation": "Literal Translation",
    "Literal Translation ": "Literal Translation",

    "Name Entity Error": "Name Entity Error",
    "Name Entity Error ": "Name Entity Error",

    "Hypernym for Hyponym": "Hypernym for Hyponym",
    "Hyponym for Hypernym": "Hyponym for Hypernym",

    "Invalid Pattern": "Invalid Pattern",
    "Tanween Omission": "Tanween Omission",
    "Gender Disagreement": "Gender Disagreement",

    "Definiteness": "Definiteness Shift",
    "Definiteness Shift": "Definiteness Shift",

    "Perfective to Progressive": "Perfective to Progressive",
    "Progressive to Perfective": "Progressive to Perfective",

    "Tense shift under negation": "Tense Shift Under Negation",
    "Tense Shift Under Negation": "Tense Shift Under Negation",
    "Negation": "Tense Shift Under Negation",

    "Wrong Structure": "Wrong Structure",
    "Wrong Word Order": "Wrong Word Order",
    "Wrong Word Order ": "Wrong Word Order",

    "Noun to Adjective": "Noun to Adjective",
    "Adjective to Noun": "Adjective to Noun",
    "Adjective to Noun ": "Adjective to Noun",

    "Active to Passive Voice": "Active to Passive Voice",
    "Passive to Active Voice": "Passive to Active Voice",

    "Formality Level": "Register Mismatch",
    "Register Mismatch": "Register Mismatch",

    "Terminology Substitution": "Terminology Substitution",
    "Spelling Error": "Spelling Error",
}


# ---------------------------------------------------------------------------
# 1. Load CSV with basic normalization
# ---------------------------------------------------------------------------

def load_gold_csv(csv_file: str = CSV_PATH) -> pd.DataFrame:
    df = pd.read_csv(csv_file, encoding="utf-8")

    # Normalize column names to a consistent key map
    cols = {c.lower().strip(): c for c in df.columns}

    def get_col(*cands):
        for c in cands:
            cl = c.lower()
            if cl in cols:
                return cols[cl]
        raise KeyError(f"Missing column, tried: {cands}")

    col_subsub = get_col("sub-subtype", "sub_subtype", "sub subtype")
    col_kw = get_col("keyword")
    col_bm = get_col("best_match", "best match")
    col_sid = get_col("sentence_id", "Sentence_ID", "id")

    df = df.rename(
        columns={
            col_subsub: "Sub-Subtype",
            col_kw: "Keyword",
            col_bm: "Best_Match",
            col_sid: "Sentence_ID",
        }
    )

    # Clean labels and strings
    df["Sub-Subtype"] = (
        df["Sub-Subtype"].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
    )

    for col in ["Keyword", "Best_Match"]:
        df[col] = (
            df[col].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
        )

    return df


# ---------------------------------------------------------------------------
# 2. Semantic validation (warnings only)
# ---------------------------------------------------------------------------

def validate_semantics(df: pd.DataFrame) -> None:
    problems = []

    # Empty Keyword is never acceptable
    mask_kw_empty = df["Keyword"].isna() | (df["Keyword"].astype(str).str.strip() == "")
    if mask_kw_empty.any():
        problems.append(f"- {mask_kw_empty.sum()} rows have empty Keyword (canonical phrase).")

    # Total Omission detection (by label, BEFORE mapping)
    mask_total = df["Sub-Subtype"].astype(str).str.strip().isin(["Total Omission", "Total Omission "])

    # For non-Total-Omission, Best_Match should not be empty
    mask_bm_empty = df["Best_Match"].isna() | (df["Best_Match"].astype(str).str.strip() == "")
    mask_non_total_bm_empty = (~mask_total) & mask_bm_empty
    if mask_non_total_bm_empty.any():
        problems.append(
            f"- {mask_non_total_bm_empty.sum()} non-'Total Omission' rows have empty Best_Match."
        )

    # For Total Omission, Best_Match can be anything; we normalize later.
    if problems:
        print("⚠️ Semantic validation warnings for 400_annotated_complete.csv:")
        for p in problems:
            print("   ", p)
        print("   (Script will continue, but you may want to inspect these rows.)")
    else:
        print("✅ Gold CSV semantics look consistent (basic checks).")


# ---------------------------------------------------------------------------
# 3. Extract raw (Sub-Subtype → {Keyword: Best_Match}) map
# ---------------------------------------------------------------------------

def extract_keywords_from_csv(
    df: pd.DataFrame,
    error_type_col: str = "Sub-Subtype",
    keyword_col: str = "Keyword",
    best_match_col: str = "Best_Match",
) -> dict:
    """
    Build a raw mapping:
        raw_error_type -> { keyword -> best_match }
    """
    keyword_map = defaultdict(dict)

    for _, row in df.iterrows():
        error_type = row[error_type_col]
        if pd.isna(error_type) or str(error_type).strip() == "":
            continue

        keyword = row[keyword_col]
        best_match = row[best_match_col]

        # Skip completely empty keyword
        if pd.isna(keyword) or str(keyword).strip() == "":
            continue

        # Best_Match may be empty only for Total Omission; we will normalize later.
        keyword = str(keyword).strip()
        best_match = "" if pd.isna(best_match) else str(best_match).strip()

        keyword_map[str(error_type)].update({keyword: best_match})

    return dict(keyword_map)


# ---------------------------------------------------------------------------
# 4. Map error-type names to canonical taxonomy
# ---------------------------------------------------------------------------

def map_error_type_names(csv_keyword_map: dict) -> dict:
    """
    Map raw 'Sub-Subtype' labels to canonical taxonomy names using PDF_TO_SCRIPT_NAMES.
    """
    mapped_keyword_map = {}

    unmapped = set()
    for raw_type, kw_dict in csv_keyword_map.items():
        script_type = PDF_TO_SCRIPT_NAMES.get(raw_type)
        if script_type:
            if script_type in mapped_keyword_map:
                mapped_keyword_map[script_type].update(kw_dict)
            else:
                mapped_keyword_map[script_type] = dict(kw_dict)
        else:
            unmapped.add(raw_type)

    if unmapped:
        print("⚠️ Unmapped Sub-Subtype labels (not in PDF_TO_SCRIPT_NAMES):")
        for t in sorted(unmapped):
            print("   -", repr(t))

    return mapped_keyword_map


# ---------------------------------------------------------------------------
# 5. Normalize Total Omission semantics in the keyword map
# ---------------------------------------------------------------------------

def normalize_total_omission(keyword_map: dict) -> dict:
    """
    For the 'Total Omission' category:
        - Ensure that all best_match values are the sentinel "[OMITTED]".
    This makes omission handling explicit and consistent for Script 3.
    """
    if "Total Omission" not in keyword_map:
        print("ℹ️ No 'Total Omission' patterns found in keyword_map.")
        return keyword_map

    om_dict = keyword_map["Total Omission"]
    new_om_dict = {}
    for kw in om_dict.keys():
        canonical_kw = str(kw).strip()
        if not canonical_kw:
            continue
        new_om_dict[canonical_kw] = "[OMITTED]"

    keyword_map["Total Omission"] = new_om_dict
    print(f"✅ Normalized Total Omission: {len(new_om_dict)} patterns set to best_match='[OMITTED]'.")
    return keyword_map


# ---------------------------------------------------------------------------
# 6. Main: build and save keyword maps
# ---------------------------------------------------------------------------

def generate_keyword_map(csv_file: str = CSV_PATH):
    print("=== Script 1: Building keyword_map.json ===")
    df = load_gold_csv(csv_file)
    print(f"Loaded CSV: {csv_file}  (rows={len(df)}, cols={len(df.columns)})")

    validate_semantics(df)

    # Step 1: raw map from CSV labels
    csv_keyword_map = extract_keywords_from_csv(df)
    print(f"Raw error types in CSV: {len(csv_keyword_map)}")

    # Step 2: map error type names to canonical labels
    keyword_map = map_error_type_names(csv_keyword_map)
    print(f"Canonical error types in keyword_map: {len(keyword_map)}")

    # Step 3: normalize Total Omission semantics
    keyword_map = normalize_total_omission(keyword_map)

    # Small stats per type
    print("\nPer-type pattern counts:")
    for etype, kw_dict in sorted(keyword_map.items()):
        print(f"  - {etype:28} {len(kw_dict):3d} pattern(s)")

    # Save keyword_map.json
    with open(KEYWORD_MAP_JSON, "w", encoding="utf-8") as f:
        json.dump(keyword_map, f, ensure_ascii=False, indent=2)
    print(f"\n✅ Saved keyword map to {KEYWORD_MAP_JSON}")

    # Also save list-of-dicts format for generation convenience
    keyword_map_for_gen = {}
    for etype, kw_dict in keyword_map.items():
        keyword_map_for_gen[etype] = [
            {"keyword": kw, "best_match": bm}
            for kw, bm in kw_dict.items()
        ]

    with open(KEYWORD_MAP_FOR_GEN_JSON, "w", encoding="utf-8") as f:
        json.dump(keyword_map_for_gen, f, ensure_ascii=False, indent=2)
    print(f"✅ Saved keyword map (for gen) to {KEYWORD_MAP_FOR_GEN_JSON}")

    print("\n=== Script 1 COMPLETE ===")


if __name__ == "__main__":
    generate_keyword_map()


In [ ]:
import json

with open("keyword_map.json", "r", encoding="utf-8") as f:
    km = json.load(f)

km_list = {
    sub: [
        {"keyword": kw, "best_match": bm}
        for kw, bm in kw_dict.items()
    ]
    for sub, kw_dict in km.items()
}

with open("keyword_map_for_gen.json", "w", encoding="utf-8") as f:
    json.dump(km_list, f, ensure_ascii=False, indent=2)
